# Data Preparation

- Feature Selection
- Feature Transfromation
- Handling Missing Values
- Handling Outliers (after outlier detection)

In [ ]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('..//dataset/cmi_internet.csv')

## 1. Feature Selection

In [ ]:
# Removing Season Columns from the Dataset

season_columns = [col for col in df.columns if 'Season' in col]

new_df = df.drop(columns=season_columns)

# Removing PCIAT columns

pci_columns = [col for col in df.columns if 'PCIAT' in col]

new_df = new_df.drop(columns=pci_columns)

# Removing Physical-BMI (= computed BMI + random noise, it is redundant)

new_df = new_df.drop(columns=['Physical-BMI'])

# Removing BIA-BIA_BMI (= it doesn't corelate with the computed BMI, it's redundant)

new_df = new_df.drop(columns=['BIA-BIA_BMI'])


## 3. Feature Transformation

In [ ]:
# Transforming Fitness_Endurance-Time_Mins in seconds and creating a new column called Fitness_Endurance-Time containing  the sum of Fitness_Endurance-Time_Mins and Fitness_Endurance-Time_Secs
new_df['Fitness_Endurance-Time'] = new_df['Fitness_Endurance-Time_Mins'] * 60 + new_df['Fitness_Endurance-Time_Secs']

new_df = new_df.drop(columns=['Fitness_Endurance-Time_Mins', 'Fitness_Endurance-Time_Secs'])

## 2. Missing Value Correction Strategy

2.1 Semantic Strategy

Filling missing values through theoretical formulae:

- BIA-BIA_TBW = BIA-BIA_ICW + BIA-BIA_ECW
- BIA-BIA_DEE = BIA-BIA_BMR * BIA-BIA_Activity_Level_num
- BIA-BIA_FFM = BIA-BIA_LST + BIA-BIA_BMC = BIA-BIA_FFMI * Height (m)^2
- BIA-BIA_FM = Physical-Weight - BIA-BIA_FFM/Physical-Weight *100 = BIA-BIA_FMI * Height (m)^2

In [ ]:
def impute_bia_values(df):
    # 1. Preparazione costanti di supporto (non sovrascriviamo le originali)
    # Convertiamo altezza in metri e peso in kg per i calcoli
    h_m = df['Physical-Height'] * 0.0254
    w_kg = df['Physical-Weight'] * 0.453592

    # --- REGOLA 1: TBW (Total Body Water) ---
    # Se manca TBW, usa la somma di ICW ed ECW
    mask_tbw = df['BIA-BIA_TBW'].isna()
    df.loc[mask_tbw, 'BIA-BIA_TBW'] = df['BIA-BIA_ICW'] + df['BIA-BIA_ECW']

    # --- REGOLA 2: FFM (Fat Free Mass) ---
    # Opzione A: Somma dei tessuti
    mask_ffm = df['BIA-BIA_FFM'].isna()
    df.loc[mask_ffm, 'BIA-BIA_FFM'] = df['BIA-BIA_LST'] + df['BIA-BIA_BMC']
    
    # Opzione B: Se manca ancora, usa l'indice FFMI (se presente)
    mask_ffm_still_na = df['BIA-BIA_FFM'].isna()
    df.loc[mask_ffm_still_na, 'BIA-BIA_FFM'] = df['BIA-BIA_FFMI'] * (h_m**2)

    # --- REGOLA 3: Fat Percentage (BIA-BIA_Fat) ---
    # Nel dataset CMI il campo è 'BIA-BIA_Fat' (percentuale %)
    mask_fat = df['BIA-BIA_Fat'].isna()
    
    # Calcoliamo prima la massa grassa (kg) in modo temporaneo
    fat_mass_kg = w_kg - df['BIA-BIA_FFM']
    # Se non abbiamo FFM, proviamo a usare l'indice FMI
    fat_mass_kg = fat_mass_kg.fillna(df['BIA-BIA_FMI'] * (h_m**2))
    
    # Convertiamo in percentuale per riempire la colonna del dataset
    df.loc[mask_fat, 'BIA-BIA_Fat'] = (fat_mass_kg / w_kg) * 100

    # --- REGOLA 4: DEE (Daily Energy Expenditure) ---
    # Nota: Usiamo i moltiplicatori standard della letteratura medica (Equazione di Harris-Benedict)
    # 1=1.2 (Sdentario), 2=1.375, 3=1.55, 4=1.725, 5=1.9 (Atleta)
    activity_map = {1: 1.2, 2: 1.375, 3: 1.55, 4: 1.725, 5: 1.9}
    multipliers = df['BIA-BIA_Activity_Level_num'].map(activity_map)
    
    mask_dee = df['BIA-BIA_DEE'].isna()
    df.loc[mask_dee, 'BIA-BIA_DEE'] = df['BIA-BIA_BMR'] * multipliers

    return df

# Esempio di utilizzo:
# df = pd.read_csv("your_dataset.csv")
# df = impute_bia_values(df)

2.2 Redundant Feature Elimination

2.3 Regression models